# **Email-Spam Classification with LoRA fine-tuning**

---

## **Table of Contents**

- [Environment Setup (Local or Google Colab)](#environment-setup-local-or-google-colab)
- [Introduction](#introduction)
- [Datasets Preparation](#datasets-preparation)
  - [Dataset Subsampling & Sequence Length Handling](#dataset-subsampling--sequence-length-handling)
  - [Download and Split Datasets](#download-and-split-datasets)
  - [Handling Long Context Sequences](#handling-long-context-sequences)
  - [Create Data Loaders](#create-data-loaders)
- [Model Setup](#model-setup)
  - [Initialize a Local GPT-2 Model and Load Weights From OpenAI](#initialize-a-local-gpt-2-model-and-load-weights-from-openai)
  - [Verify The Loaded Model](#verify-the-loaded-model)
- [Finetuning The Model](#finetuning-the-model)
  - [Upgrading the Architecture with LoRA](#upgrading-the-architecture-with-lora)
  - [Classification Head Adaptation](#classification-head-adaptation)
  - [Sanity Check: Output Shape & Logit Verification](#sanity-check-output-shape--logit-verification)
  - [Pre-Fine-Tuning Baseline Evaluation](#pre-fine-tuning-baseline-evaluation)
  - [Fine-Tuning Execution & Optimizer Setup](#fine-tuning-execution--optimizer-setup)
  - [Visualization of Training Dynamics](#visualization-of-training-dynamics)
- [Final Metric Evaluation Across Datasets](#final-metric-evaluation-across-datasets)
  - [Performance Analysis & Observations](#performance-analysis--observations)

---

## **Environment Setup (Local or Google Colab)**

The code block below automatically detects your execution runtime environment (Local vs. Google Colab), clones the repository, and installs all necessary dependencies.

> **Google Colab Note:** Once setup completes, navigate to **Runtime → Restart session** in the top menu before proceeding to the next steps to ensure all newly installed packages are correctly loaded.

In [1]:
import os
import subprocess
import sys

import torch

# Environment Detection: Check sys.modules for Colab runtime
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Detected Google Colab environment. Initializing setup...")

    # Define paths
    branch = "dev"
    repo_url = (
        "https://github.com/paymantohidifar/gpt2-text-classifier-from-scratch.git"
    )
    target_dir = "/content/gpt2-classifier"

    # Clean stale builds and clone main branch
    subprocess.run(f"rm -rf {target_dir}", shell=True, check=True)
    subprocess.run(
        f"git clone {repo_url} --branch {branch} {target_dir}",
        shell=True,
        check=True,
    )
    print("Cloned the repo.")

    # Change working directory
    os.chdir(target_dir)

    # Bootstrap uv and install dependencies into system environment
    install_cmd = (
        'curl -LsSf https://astral.sh/uv/install.sh | sh && '
        'export PATH="$HOME/.local/bin:${PATH}" && '
        'uv pip install -e .[gpu,dev] --system --break-system-packages --color never'
    )
    print("Installing dependencies...")
    subprocess.run(install_cmd, shell=True, check=True)
    print("Colab setup complete.")

else:
    # Local Development: Enable IPython Auto-Reload safely
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic("load_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")
        print("Enabled IPython autoreload.")


# Verify PyTorch version and config
print(f"\nInstalled Pytorch version: {torch.__version__}")
print(f"PyTorch enabled with Cuda: {torch.cuda.is_available()}")

Detected Google Colab environment. Initializing setup...
Cloned the repo.
Installing dependencies...
Colab setup complete.

Installed Pytorch version: 2.11.0+cu128
PyTorch enabled with Cuda: True


---

## **Introduction**

This notebook is a reimplementation of the [02_email_spam_classification](./02_email_spam_classification.ipynb), upgrading our optimization strategy to use **Low-Rank Adaptation (LoRA)** ([Hu et al., 2021](https://arxiv.org/abs/2106.09685)).

This parameter-efficient fine-tuning (PEFT) method drastically reduces VRAM consumption, compute requirements, and storage overhead, enabling us to store a single base model alongside multiple lightweight, task-specific LoRA adapters.

We will fine-tune a custom GPT-2 architecture to categorize emails as **ham** or **spam** using the open-source [Email Spam Collection](https://raw.githubusercontent.com/MWiechmann/enron_spam_data/master/enron_spam_data.zip). In our previous baseline, we only unfroze the terminal `LayerNorm`, the final transformer block, and the classification head. Here, we will inject LoRA adapters to fine-tune all linear projection layers across the entire transformer backbone except for output head, which we will keep full linear layer for improved learning.

All core modules and utilities are imported from the custom `gpt2_classifier` package. For production deployments, this end-to-end training and inference pipeline can be executed via the command-line interface (CLI). Refer to the [README.md](../README.md) for configuration flags.

---

## **Datasets Preparation**

The raw **Enron Spam** dataset has balanced class labels (~50% spam vs. ~50% ham). In this section, we download and construct training, validation, and test sets with a consistent 70%/10%/20% split using the `prepare_dataset` function.

### **Dataset Subsampling & Sequence Length Handling**

The **Enron Email Spam** dataset is substantially larger than the SMS Spam collection (33,716 samples). To accommodate hardware memory constraints and accelerate training iterations, we apply a 10% random subsampling factor via the `hold_frac` configuration parameter in the `prepare_dataset` function.

> **Trade-off Note:** Subsampling significantly reduces compute overhead, but shrinks overall dataset coverage, which may increase the risk of model overfitting.

### **Download and Split Datasets**

In [1]:
from gpt2_classifier.data import prepare_dataset
from gpt2_classifier.datasets_registry import get_dataset_spec

spec = get_dataset_spec("email-spam")
print(spec)

path = prepare_dataset(spec, hold_frac=0.1, force=True, balance_labels=False, dataset_split=(0.7, 0.1, 0.2))
print("File path:", path)

DatasetSpec(name='email-spam', url='https://raw.githubusercontent.com/MWiechmann/enron_spam_data/master/enron_spam_data.zip', raw_filename='enron_spam_data.csv', text_column='Message', label_column='Spam/Ham', label_map={'ham': 0, 'spam': 1}, archive_format='zip', separator=',', has_header=True, column_names=None, backup_url=None)
File downloaded and saved as /content/gpt2-classifier/data/email-spam/enron_spam_data.csv
File path: /content/gpt2-classifier/data/email-spam


### **Handling Long Context Sequences**

Email sequences are substantially longer than SMS messages (averaging 30,000–50,000 tokens per email vs. ~120 tokens for SMS). This drastically exceeds GPT-2's native context window limit of **1,024 tokens**.

While advanced strategies exist, such as **Rotary Position Embeddings (RoPE) scaling** or **sliding-window chunking** (splitting single long emails into $K$ overlapping batches), they fall outside the scope of this notebook.

For simplicity and execution speed, we will cap the context size to model's maximum context window size of 1024 directly within our `create_data_loaders` pipeline to truncate sequences. In the evaluation phase, we will analyze how this context truncation impacts model performance metrics.

### **Create Data Loaders**

We construct data loaders for the train, validation, and test splits to manage mini-batching during optimization. By default, we set `batch_size = 8` and enable shuffling on the training set to ensure mini-batches provide unbiased gradient estimates and prevent catastrophic order-dependent learning. As discussed above, we also set the context size `max_length=1024`.

*Note: While batch size is a tunable hyperparameter, scaling it up increases per-step memory consumption and may trigger Out-Of-Memory (OOM) errors depending on your GPU's VRAM capacity.*

In [2]:
from gpt2_classifier.data import create_data_loaders

train_loader, val_loader, test_loader = create_data_loaders(spec.name, max_length=1024, batch_size=8)

for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

Input batch dimensions: torch.Size([8, 1024])
Label batch dimensions torch.Size([8])
293 training batches
42 validation batches
84 test batches


---

## **Model Setup**

With our data loaders fully configured, we instantiate our custom GPT-2 architecture. Training a transformer language model from scratch requires millions of GPU hours to converge on meaningful semantic representations. To bypass this compute barrier, we populate our model’s parameters directly with OpenAI’s pretrained weights, retrieved using Hugging Face's safetensors API. This process is managed by two core utilities: `download_and_load_gpt2` (weight retrieval) and `load_weights_into_gpt` (state dict mapping).

For this notebook, we instantiate the base GPT-2 Small (124M parameters) model, which provides a strong capacity-to-compute ratio for our target dataset. You can experiment with larger variants by updating the `model_name` configuration string:

* `gpt2-small (124M)`
* `gpt2-medium (355M)`
* `gpt2-large (774M)`
* `gpt2-xl (1558M)`

*Note: Scaling up model size increases activation and optimizer memory footprints exponentially, requiring longer per-step latency and higher VRAM capacity.*

### **Initialize a Local GPT-2 Model and Load Weights From OpenAI**

In [3]:
from gpt2_classifier import paths
from gpt2_classifier.config import URL_DIR, get_model_config
from gpt2_classifier.model import GPTModel
from gpt2_classifier.weights import download_and_load_gpt2, load_weights_into_gpt

model_name = "gpt2-small (124M)"
model_config = get_model_config(model_name)
model_url = f"https://huggingface.co/openai-community/{URL_DIR[model_name]}/resolve/main/model.safetensors"
model_destination = paths.MODELS_DIR / f"{URL_DIR[model_name]}.safetensors"

state_dict = download_and_load_gpt2(model_url, model_destination)
model = GPTModel(model_config)
load_weights_into_gpt(model, state_dict)

print(model)

model.safetensors: 100%|██████████| 548M/548M [00:07<00:00, 74.4MiB/s]


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

### **Verify The Loaded Model**

To verify that the pretrained weight transfer succeeded and that our model state is correctly configured, we perform a qualitative sanity check by generating text from a prompt. Coherent text output confirms proper weight mapping across transformer layers, whereas unconditioned noise or gibberish indicates a state dict misalignment or missing layer keys.

In [4]:
from gpt2_classifier.utils import generate_response

text_1 = "Every effort moves you"

reponse = generate_response(
    text=text_1,
    model=model,
    max_new_tokens=15
)

print(reponse)

Every effort moves you forward.

The first step is to understand the importance of your work


---

## **Finetuning The Model**


### **Upgrading the Architecture with LoRA**

Instead of updating a massive, full-rank weight matrix $W_0 \in \mathbb{R}^{d_{\text{in}} \times d_{\text{out}}}$, Low-Rank Adaptation (LoRA) freezes the pretrained weights and parameterizes the update matrix $\Delta W$ using a low-rank decomposition:

$$\Delta W = A B$$

where $A \in \mathbb{R}^{d_{\text{in}} \times r}$ and $B \in \mathbb{R}^{r \times d_{\text{out}}}$. By setting the rank $r \ll \min(d_{\text{in}}, d_{\text{out}})$, we constrain parameter updates to a lower-dimensional subspace. Thanks to the distributive property of matrix multiplication, the forward pass dynamically combines the frozen and trainable representations:

$$h = x W_0 + x \Delta W = x W_0 + x A B$$

Next, we wrap every target linear layer with a LoRA adapter. To build intuition, we will manually construct the low-rank layers and swap them into the transformer backbone. In production workflows, this step can be automated using the `replace_linear_with_lora` utility from the `pfdl.train` module.

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable parameters = ", total_params)

# Freeze the orignal model parameters
for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable parameters after freezing the model= ", total_params)

In [ ]:
import torch
from torch import nn
import math

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.rank = rank
        self.alpha = alpha

    def forward(self, x):
        return (self.alpha / self.rank) * (x @ self.A @ self.B)


class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)


def replace_linear_for_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_for_lora(module, rank, alpha)

In [ ]:
# rank is a hyperparameter.
# Common rank/alpha is 16
rank = 16
alpha = 16
replace_linear_for_lora(model, rank, alpha)

# Print number of trainable parameters after LoRA upgrade
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable parameters with LoRA = ", total_params)

# Alternatively, we could directly use `replace_linear_for_lora`
# from gpt2_classifier.train import replace_linear_for_lora

# rank = 16
# alpha = 16
# replace_linear_for_lora(model, rank, alpha)

### **Classification Head Adaptation**

The standard GPT-2 language modeling head projects transformer hidden states to a vocabulary dimension of 50,257 logits. For our binary classification task, we replace this final projection layer with a linear head mapping the output dimension from $d_{\text{model}}$ to 2 class logits.

In [ ]:
import torch

# Swap teh GPT-2 head with a binary head
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(in_features=model_config['emb_dim'], out_features=num_classes)

print(model)

### **Sanity Check: Output Shape & Logit Verification**

We perform a quick forward pass using a toy data to verify that our modified architecture correctly outputs a tensor of shape `(batch_size, 2)`. Confirming that the classification head returns two unnormalized logits ensures that our state dict modifications and layer swaps align with the binary classification loss formulation prior to starting fine-tuning.

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print(inputs)

device = "cpu"
inputs = inputs.to(device)

with torch.no_grad():
    outputs = model(inputs)
print(outputs)
print(outputs.shape)

### **Pre-Fine-Tuning Baseline Evaluation**

Before initiating parameter updates, we evaluate our initialized model across the training, validation, and test splits to establish benchmark loss and accuracy metrics. Recording pre-training baselines serves two critical functions: it quantifies the exact performance gains attributable to fine-tuning and acts as a diagnostic check to ensure loss decreases monotonically from a reasonable starting point.

In [8]:
from gpt2_classifier.evaluate import calc_accuracy_loader, calc_loss_loader
from gpt2_classifier.utils import get_device

device = get_device()
model.to(device)

torch.manual_seed(123)

# Compute accuracy for all three datasets
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")


# Compute losses for all three datasets
with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
    test_loss = calc_loss_loader(test_loader, model, device, num_batches=5)

print(f"Training loss: {train_loss:.3f}")
print(f"Validation loss: {val_loss:.3f}")
print(f"Test loss: {test_loss:.3f}")

Training accuracy: 40.00%
Validation accuracy: 55.00%
Test accuracy: 56.25%
Training loss: 4.247
Validation loss: 3.699
Test loss: 3.579


### **Fine-Tuning Execution & Optimizer Setup**

With baseline benchmarks established, we can now proceed to fine-tune our classifier. Before launching the training loop, we must instantiate our optimization algorithm. We utilize the AdamW optimizer for model training. For a concise breakdown of Adam versus AdamW (specifically regarding decoupled weight decay), please refer to the [01_sms_spam_classification](https://colab.research.google.com/github/paymantohidifar/gpt2-text-classifier-from-scratch/blob/main/notebooks/01_sms_spam_classification.ipynb) notebook.

Due to the expanded sequence context window and the larger volume of training examples, fine-tuning will take significantly longer, approximately 11-12 minutes per epoch on an Nvidia T4 GPU via Google Colab.

In [11]:
import time

from gpt2_classifier.train import get_adam_param_groups, train_classifier_simple
from gpt2_classifier.utils import get_device

device = get_device()
model.to(device)

start_time = time.time()

torch.manual_seed(123)
# optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
optim_groups = get_adam_param_groups(model, weight_decay=0.1)
optimizer = torch.optim.AdamW(optim_groups, lr=5e-5)

num_epochs = 5

history = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=100, eval_iter=10,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Ep 1 (Step 000000): Train loss 4.065, Val loss 3.421
Ep 1 (Step 000100): Train loss 0.986, Val loss 0.763
Ep 1 (Step 000200): Train loss 0.805, Val loss 0.743
Training accuracy: 53.75% | Validation accuracy: 53.75%
Train precision: 0.483, ROC-AUC: 0.590, PR-AUC: 0.496 | Val precision: 0.625, ROC-AUC: 0.597, PR-AUC: 0.610
Ep 2 (Step 000300): Train loss 0.770, Val loss 0.656
Ep 2 (Step 000400): Train loss 0.642, Val loss 0.643
Ep 2 (Step 000500): Train loss 0.718, Val loss 0.624
Training accuracy: 52.50% | Validation accuracy: 60.00%
Train precision: 0.522, ROC-AUC: 0.622, PR-AUC: 0.624 | Val precision: 0.587, ROC-AUC: 0.647, PR-AUC: 0.702
Ep 3 (Step 000600): Train loss 0.653, Val loss 0.643
Ep 3 (Step 000700): Train loss 0.519, Val loss 0.566
Ep 3 (Step 000800): Train loss 0.548, Val loss 0.563
Training accuracy: 66.25% | Validation accuracy: 71.25%
Train precision: 0.609, ROC-AUC: 0.779, PR-AUC: 0.739 | Val precision: 0.667, ROC-AUC: 0.771, PR-AUC: 0.763
Ep 4 (Step 000900): Train loss 

We could also use `finetune_model` function to automatically set up the optimizer, train the model, and checkpoint the trained model.

### **Visualization of Training Dynamics**

Below, we plot the trajectory of our primary training and validation metrics across epochs.

*Note: While our CLI script supports the `--use-wandb` flag for real-time cloud tracking of loss curves, evaluation metrics, and hardware utilization (GPU/VRAM throughput), we use static inline visualizations here for standalone notebook execution.*

In [ ]:
from gpt2_classifier.utils import plot_results

fig_path = plot_results(
    num_epochs, *history,
    dataset="sms-spam",
    title="GPT-2 classifier with balance sms-spam data"
    )

print(fig_path)

---

## **Final Metric Evaluation Across Datasets**

As a final step in our evaluation pipeline, we compute comprehensive classification metrics across the training, validation, and test splits. Aggregating performance over all dataset mini-batches provides a statistically robust, out-of-sample assessment of our fine-tuned model's generalization capability.

In [ ]:
from gpt2_classifier.evaluate import calc_classification_metrics_loader

train_metrics = calc_classification_metrics_loader(train_loader, model, device, num_batches=300)
val_metrics = calc_classification_metrics_loader(val_loader, model, device, num_batches=300)
test_metrics = calc_classification_metrics_loader(test_loader, model, device, num_batches=300)

print("Training metrics:")
print(
    f"Accuracy: {train_metrics.accuracy:.3f} | Precision: {train_metrics.precision:.3f} | ROC-AUC: {train_metrics.roc_auc:.3f}, |"
    f"PR-AUC: {train_metrics.pr_auc:.3f}\n"
)

print("Validation metrics:")
print(
    f"Accuracy: {val_metrics.accuracy:.3f} | Precision: {val_metrics.precision:.3f} | ROC-AUC: {val_metrics.roc_auc:.3f}, |"
    f"PR-AUC: {val_metrics.pr_auc:.3f}\n"
)

print("Test metrics:")
print(
    f"Accuracy: {test_metrics.accuracy:.3f} | Precision: {test_metrics.precision:.3f} | ROC-AUC: {test_metrics.roc_auc:.3f}, |"
    f"PR-AUC: {test_metrics.pr_auc:.3f}\n"
)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Combine all metrics for Train, Validation, and Test sets in one dataframe
metrics_df = pd.DataFrame(
    {
        'Train': dict(train_metrics),
        'Validation': dict(val_metrics),
        'Test': dict(test_metrics),
    },
    index=pd.Index(['accuracy', 'precision', 'roc_auc', 'pr_auc'], name='Metric')
).reset_index()

# Pivot dataframe suitable for plotting
metrics_df_long = pd.melt(
    metrics_df, id_vars=['Metric'], value_vars=['Train', 'Validation', 'Test'], var_name='Set', value_name='Value')

fig, ax = plt.subplots(figsize=(5,4))

sns.barplot(ax=ax, data=metrics_df_long, x='Metric', y='Value', hue='Set')
ax.set_title("Classification Metrics")
ax.set_xlabel("Metric")
ax.set_ylabel("Value")
ax.legend(title="Set", loc="lower right")
plt.show()
fig_dir  = paths.PLOTS_DIR / spec.name
if not fig_dir.exists():
    fig_dir.mkdir(parents=True)
fig.savefig(fig_dir / "model_metrics.png", dpi=300, bbox_inches='tight');

### **Performance Analysis & Observations**

Across this training run, we observe consistent improvements across all metrics; however, overall performance remains below target expectations. Key observations include:

* **Overfitting**: There are currently no signs of overfitting at this epoch. Validation loss continues to closely track training loss.

* **Convergence Rate**: The model shows steady progress, suggesting that training for additional epochs could yield further gains. However, the loss profile indicates that convergence is slowing down and performance improvements will likely be gradual.

* **Primary Bottleneck**: A primary contributor to the lower-than-expected performance is context window truncation. Due to memory limits and max sequence length constraints, we are training on only a sub-fraction of the available text tokens per example, omitting valuable contextual signals.